# 09 — SBERT Fine-Tuning for IFRS Taxonomy Matching

Fine-tunes Sentence-BERT (all-MiniLM-L6-v2) with contrastive
learning on financial line-item → IFRS taxonomy code pairs.
Uses training data from XBRL auto-labeling (notebook 03).

**Target accuracy@1**: 85%+

In [ ]:
# Cell 1: Setup
!pip install -q sentence-transformers scikit-learn tqdm PyYAML mlflow

import json, os, random
from pathlib import Path
import torch
import yaml
import numpy as np
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
    util,
)
from torch.utils.data import DataLoader
import mlflow

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
MODELS_DIR = Path(cfg['drive']['models_dir'])
scfg = cfg['sbert']

model = SentenceTransformer(scfg['model_name'])
print(f'Model: {scfg["model_name"]} loaded ✅')

In [ ]:
# Cell 2: Load training pairs and taxonomy
pairs_file = DATA_DIR / 'labels' / 'sbert_training_pairs.json'
with open(pairs_file) as f:
    all_pairs = json.load(f)

# Load taxonomy for negative sampling
taxonomy_file = Path('../../data/ifrs_taxonomy.json')
if not taxonomy_file.exists():
    taxonomy_file = DATA_DIR / 'ifrs_taxonomy.json'
with open(taxonomy_file) as f:
    taxonomy = json.load(f)

taxonomy_items = []
if isinstance(taxonomy, dict) and 'elements' in taxonomy:
    taxonomy_items = taxonomy['elements']
elif isinstance(taxonomy, list):
    taxonomy_items = taxonomy

taxonomy_labels = [t.get('label', t.get('name', '')) for t in taxonomy_items]

# Train/val split
random.seed(42)
random.shuffle(all_pairs)
split = int(len(all_pairs) * cfg['data']['train_split'])
train_pairs = all_pairs[:split]
val_pairs = all_pairs[split:]

print(f'Train: {len(train_pairs)}, Val: {len(val_pairs)}, Taxonomy: {len(taxonomy_labels)}')

In [ ]:
# Cell 3: Create training examples with hard negatives

def find_label_by_code(code: str, items: list) -> str:
    for item in items:
        if item.get('code', item.get('id', '')) == code:
            return item.get('label', item.get('name', ''))
    return code  # fallback

def create_training_examples(pairs, taxonomy_labels, neg_ratio=3):
    """Create InputExamples with positive pairs and hard negatives."""
    examples = []
    
    for pair in pairs:
        query = pair['text']
        positive = find_label_by_code(pair['taxonomy_code'], taxonomy_items)
        
        # Positive pair (label=1.0)
        examples.append(InputExample(texts=[query, positive], label=1.0))
        
        # Hard negatives: random taxonomy items that aren't the positive
        negatives = random.sample(
            [t for t in taxonomy_labels if t != positive],
            min(neg_ratio, len(taxonomy_labels) - 1),
        )
        for neg in negatives:
            examples.append(InputExample(texts=[query, neg], label=0.0))
    
    random.shuffle(examples)
    return examples

train_examples = create_training_examples(train_pairs, taxonomy_labels)
print(f'Training examples: {len(train_examples)} (pos + neg)')

In [ ]:
# Cell 4: Create evaluator

# Build evaluation set
eval_sentences1 = [p['text'] for p in val_pairs]
eval_sentences2 = [find_label_by_code(p['taxonomy_code'], taxonomy_items) for p in val_pairs]
eval_scores = [1.0] * len(val_pairs)

evaluator = evaluation.EmbeddingSimilarityEvaluator(
    eval_sentences1, eval_sentences2, eval_scores,
    name='ifrs-val',
    show_progress_bar=True,
)

# Test baseline
baseline_score = evaluator(model)
print(f'Baseline similarity score: {baseline_score:.4f}')

In [ ]:
# Cell 5: Fine-tune
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=scfg['batch_size'])
train_loss = losses.CosineSimilarityLoss(model)

# Configure MLflow
mlflow.set_tracking_uri(cfg['mlflow']['tracking_uri'])
mlflow.set_experiment(scfg['mlflow_experiment'])

warmup_steps = int(len(train_dataloader) * scfg['num_epochs'] * scfg['warmup_ratio'])
output_path = str(MODELS_DIR / 'sbert-ifrs-finetuned')

print(f'Training: {scfg["num_epochs"]} epochs, LR: {scfg["learning_rate"]}, Warmup: {warmup_steps}')

with mlflow.start_run(run_name='sbert-ifrs-v1'):
    mlflow.log_params(scfg)
    
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        evaluator=evaluator,
        epochs=scfg['num_epochs'],
        warmup_steps=warmup_steps,
        output_path=output_path,
        evaluation_steps=scfg['evaluation_steps'],
        save_best_model=True,
        optimizer_params={'lr': scfg['learning_rate']},
    )
    
    print(f'\nTraining complete! Model saved to: {output_path}')

In [ ]:
# Cell 6: Evaluate fine-tuned model
finetuned_model = SentenceTransformer(output_path)

# Retrieval evaluation (same as notebook 08)
taxonomy_embeddings = finetuned_model.encode(taxonomy_labels, convert_to_tensor=True)
taxonomy_codes = [t.get('code', t.get('id', '')) for t in taxonomy_items]

queries = [p['text'] for p in val_pairs]
true_codes = [p['taxonomy_code'] for p in val_pairs]
query_embeddings = finetuned_model.encode(queries, convert_to_tensor=True)
scores = util.cos_sim(query_embeddings, taxonomy_embeddings)

acc1 = acc5 = mrr_sum = 0
for i, (sc, tc) in enumerate(zip(scores, true_codes)):
    top = torch.argsort(sc, descending=True)
    top_codes = [taxonomy_codes[idx] for idx in top[:10]]
    if tc in top_codes[:1]: acc1 += 1
    if tc in top_codes[:5]: acc5 += 1
    for rank, code in enumerate(top_codes, 1):
        if code == tc:
            mrr_sum += 1.0 / rank
            break

n = len(val_pairs)
print('\n=== Fine-Tuned SBERT Results ===')
print(f'  Accuracy@1: {acc1/n:.4f}')
print(f'  Accuracy@5: {acc5/n:.4f}')
print(f'  MRR: {mrr_sum/n:.4f}')

# Compare with baseline
baseline_file = DATA_DIR / 'sbert_baseline_results.json'
if baseline_file.exists():
    with open(baseline_file) as f:
        baseline = json.load(f)
    print(f'\n  Improvement over baseline:')
    print(f'    Accuracy@1: {baseline.get("accuracy@1", 0):.4f} → {acc1/n:.4f}')
    print(f'    MRR: {baseline.get("mrr", 0):.4f} → {mrr_sum/n:.4f}')